In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [2]:
data = pd.read_csv("../01 Datasets\Churn_Modelling.csv")

In [3]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

In [5]:
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])

In [6]:
onehot_encoder_geo = OneHotEncoder(handle_unknown="ignore")
geo_encoded = onehot_encoder_geo.fit_transform(data[["Geography"]]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(["Geography"])
)

In [7]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [8]:
data = pd.concat([data.drop("Geography", axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [9]:
X = data.drop("EstimatedSalary", axis=1)
y = data["EstimatedSalary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(label_encoder_gender, file)

with open("one_hot_geo.pkl", "wb") as file:
    pickle.dump(onehot_encoder_geo, file)

with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

# ANN Regression Problem Statement

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [13]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)
])

In [14]:
model.compile(optimizer='adam', loss = 'mean_absolute_error', metrics=['mae'])


In [15]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_1 (Dense)             (None, 64)                832       
                                                                 
 dense_2 (Dense)             (None, 32)                2080      
                                                                 
 dense_3 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [20]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import warnings

warnings.filterwarnings("ignore")

In [21]:
## set up the Tensorboard
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [22]:
## set up EarlyStopping
early_stopping_callback = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [24]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[tensorboard_callback, early_stopping_callback],
)

Epoch 1/100


250/250 [==============================] - 7s 11ms/step - loss: 100378.2969 - mae: 100378.2969 - val_loss: 98520.0391 - val_mae: 98520.0391
Epoch 2/100
250/250 [==============================] - 2s 7ms/step - loss: 99621.9766 - mae: 99621.9766 - val_loss: 96989.7656 - val_mae: 96989.7656
Epoch 3/100
250/250 [==============================] - 2s 10ms/step - loss: 96924.1172 - mae: 96924.1172 - val_loss: 93050.6797 - val_mae: 93050.6797
Epoch 4/100
250/250 [==============================] - 2s 7ms/step - loss: 91619.6172 - mae: 91619.6172 - val_loss: 86460.0938 - val_mae: 86460.0938
Epoch 5/100
250/250 [==============================] - 1s 6ms/step - loss: 83887.0547 - mae: 83887.0547 - val_loss: 77961.9922 - val_mae: 77961.9922
Epoch 6/100
250/250 [==============================] - 2s 8ms/step - loss: 74808.8906 - mae: 74808.8906 - val_loss: 69049.7891 - val_mae: 69049.7891
Epoch 7/100
250/250 [==============================] - 3s 11ms/step - loss: 65974.6719 - mae: 65974.

In [25]:
%load_ext tensorboard

In [29]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6006 (pid 1396), started 0:00:07 ago. (Use '!kill 1396' to kill it.)

In [30]:
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f'Test MAE: {test_mae}')

63/63 [==============================] - 0s 3ms/step - loss: 50317.3164 - mae: 50317.3164
Test MAE: 50317.31640625


In [31]:
model.save('regressionmodel.h5')